# Variational autoencoder

Same widths as the vanilla autoencoder, but the 64-d code is a sample from `N(mu, sigma)` and the loss adds a KL term that pulls that distribution toward `N(0, I)`. Reconstruction uses the mean code. New images are drawn by decoding prior noise, which feeds Figure 2 and the FID column.

Run this on a Colab **GPU** runtime. The project folder must sit at `MyDrive/dl-sesi07`. Change `DRIVE_ROOT` in the setup cell if your folder name differs. Run the cells from top to bottom. Results are written to `artifacts/` on Drive.


In [ ]:
import os
import sys
from pathlib import Path

DRIVE_ROOT = "/content/drive/MyDrive/dl-sesi07"

if "google.colab" in sys.modules:
    from google.colab import drive
    drive.mount("/content/drive")
    ROOT = Path(DRIVE_ROOT)
else:
    ROOT = Path.cwd().resolve()
    if ROOT.name == "notebooks":
        ROOT = ROOT.parent

os.chdir(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

if "google.colab" in sys.modules:
    import subprocess
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"])


## Setup

Pixels stay in `[0, 1]`. The decoder ends in a sigmoid.


In [ ]:
%matplotlib inline
import time

import torch
import torch.nn.functional as F
from torch.optim import Adam
from tqdm.auto import tqdm

from src.config import (
    ALPHAS,
    CLASS_NAMES,
    FID_SAMPLES,
    GRID_SAMPLES,
    LATENT_DIM,
    SEED,
    get_device,
    set_seed,
    standard_normal,
)
from src.data import class_pair, fixed_test_images, get_splits, make_loader
from src.io import load_checkpoint, load_json, save_json, save_shared, save_torch
from src.metrics import measure_latency_ms, reconstruction_scores, stability_from_curve
from src.models import VAE
from src.plots import show_grid

set_seed(SEED)
DEVICE = get_device()
EPOCHS = 30
NAME = "vae"
print("device", DEVICE)

train_set, _, test_set = get_splits()
test_loader = make_loader(test_set, shuffle=False)
test_images, test_labels = fixed_test_images()
save_shared("test8.pt", {"images": test_images, "labels": test_labels})

def latent_strip(decode, z_a, z_b):
    frames = []
    for alpha in ALPHAS:
        frames.append(decode((1 - alpha) * z_a + alpha * z_b).cpu())
    return torch.cat(frames)


## Loss

Per image, the objective is the sum of pixel MSE plus the KL between `q(z|x)` and `N(0, I)`. Summing the pixels keeps the KL on a comparable scale. The plotted reconstruction curve, and the Test MSE, are still the mean per pixel. `z = mu + eps * sigma` is the reparameterization.


In [ ]:
model = VAE().to(DEVICE)
optimizer = Adam(model.parameters(), lr=1e-3)
history = {"recon_loss": [], "kl": [], "epoch_seconds": []}

for epoch in range(EPOCHS):
    set_seed(SEED + epoch)
    loader = make_loader(train_set, shuffle=True, seed=SEED + epoch)
    model.train()
    total_mse = 0.0
    total_kl = 0.0
    seen_pixels = 0
    seen_images = 0
    start = time.perf_counter()
    for images, _ in tqdm(loader, leave=False):
        images = images.to(DEVICE)
        restored, mu, logvar = model(images)
        pixel_sum = F.mse_loss(restored, images, reduction="sum")
        recon_per_image = pixel_sum / images.size(0)
        kl = -0.5 * torch.sum(1 + logvar - mu.pow(2) - logvar.exp()) / images.size(0)
        loss = recon_per_image + kl
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        total_mse += pixel_sum.item()
        total_kl += kl.item() * images.size(0)
        seen_pixels += images.numel()
        seen_images += images.size(0)
    elapsed = time.perf_counter() - start
    history["recon_loss"].append(total_mse / seen_pixels)
    history["kl"].append(total_kl / seen_images)
    history["epoch_seconds"].append(elapsed)
    print(f"epoch {epoch + 1:02d}  mse {total_mse / seen_pixels:.4f}  kl {total_kl / seen_images:.2f}  {elapsed:.1f}s")

save_json(NAME, "history.json", history)
save_torch(NAME, "checkpoint.pt", {"model": model.state_dict(), "epoch": EPOCHS - 1})


## Scores, prior samples, interpolation

Reconstruction decodes `mu`, not a fresh sample, so the test score is stable. The 16-image grid and the 5,000 FID images decode `z ~ N(0, I)`. Latency times that decoder on a batch of 64, which is the single-pass generator in the trilemma comparison.


In [ ]:
checkpoint = load_checkpoint(NAME, map_location=DEVICE)
model.load_state_dict(checkpoint["model"])
model.eval()
history = load_json(NAME, "history.json")

def reconstruct(images):
    mu, _ = model.encode(images)
    return model.decode(mu)

scores = reconstruction_scores(reconstruct, test_loader, DEVICE)
preview = reconstruct(test_images.to(DEVICE)).cpu()

image_a, image_b = class_pair()
with torch.no_grad():
    mu_a, _ = model.encode(image_a.to(DEVICE))
    mu_b, _ = model.encode(image_b.to(DEVICE))
    strip = latent_strip(model.decode, mu_a, mu_b)
    grid = model.decode(standard_normal(GRID_SAMPLES, LATENT_DIM, SEED).to(DEVICE)).cpu()

set_seed(SEED)
fid_chunks = []
left = FID_SAMPLES
with torch.no_grad():
    while left:
        count = min(128, left)
        noise = torch.randn(count, LATENT_DIM, device=DEVICE)
        fid_chunks.append(model.decode(noise).cpu())
        left -= count
fid_images = torch.cat(fid_chunks)

noise_64 = standard_normal(64, LATENT_DIM, SEED).to(DEVICE)
latency = measure_latency_ms(lambda: model.decode(noise_64), DEVICE)

save_torch(NAME, "recon.pt", preview)
save_torch(NAME, "interp.pt", strip)
save_torch(NAME, "samples_grid.pt", grid)
save_torch(NAME, "samples_fid.pt", fid_images)
save_json(
    NAME,
    "metrics.json",
    {
        "latent_dim": LATENT_DIM,
        "sec_per_epoch": sum(history["epoch_seconds"]) / len(history["epoch_seconds"]),
        "latency_ms": latency,
        "test_mse": scores["test_mse"],
        "test_psnr": scores["test_psnr"],
        "test_ssim": scores["test_ssim"],
        "stability": stability_from_curve(history["recon_loss"]),
    },
)
show_grid(torch.cat([test_images, preview]), nrow=8, title="Originals, then mean-code reconstructions")
show_grid(grid, nrow=4, title="Prior samples")
print({key: round(value, 4) if isinstance(value, float) else value for key, value in load_json(NAME, "metrics.json").items()})
